# Serving a Model with FastAPI

> Expose a small prediction pipeline through a typed HTTP endpoint and test it locally.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Understand model serving and the request/response contract.
- Build a minimal FastAPI application around a fitted model.
- Validate input shape and return explicit outputs.

This notebook defines the API object and tests the route in-process; it does not launch a public server. Production serving also needs authentication/authorization, timeouts, logging, monitoring, dependency pinning, and deployment configuration.

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, _, y_train, _ = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
FEATURES = list(X.columns)
app = FastAPI(title="ML Demo API", version="1.0.0")

class PredictionRequest(BaseModel):
    features: dict[str, float]

class PredictionResponse(BaseModel):
    predicted_class: int
    probability_class_1: float

In [2]:
@app.get("/health")
def health():
    return {"status": "ok", "model_version": "demo-1.0.0"}

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    missing = sorted(set(FEATURES) - set(request.features))
    extra = sorted(set(request.features) - set(FEATURES))
    if missing or extra:
        raise HTTPException(status_code=422, detail={"missing_features": missing, "unexpected_features": extra})
    row = pd.DataFrame([[request.features[name] for name in FEATURES]], columns=FEATURES)
    probability = float(model.predict_proba(row)[0, 1])
    label = int(probability >= .5)
    return PredictionResponse(predicted_class=label, probability_class_1=probability)

print("Routes:", [(route.path, sorted(route.methods or [])) for route in app.routes])

Routes: [('/openapi.json', ['GET', 'HEAD']), ('/docs', ['GET', 'HEAD']), ('/docs/oauth2-redirect', ['GET', 'HEAD']), ('/redoc', ['GET', 'HEAD']), ('/health', ['GET']), ('/predict', ['POST'])]


## Calling the API
With FastAPI and Uvicorn installed, save the app as `main.py` and run `uvicorn main:app --reload`. The interactive docs are available at `/docs` on localhost. Do not expose a demo endpoint publicly without deployment hardening.

In [3]:
# Build a valid example payload from one record.
example = X.iloc[0].to_dict()
response = predict(PredictionRequest(features={key: float(value) for key,value in example.items()}))
print(response.model_dump())
print(health())

{'predicted_class': 0, 'probability_class_1': 6.1181068423414845e-09}
{'status': 'ok', 'model_version': 'demo-1.0.0'}


## Production checklist
- Validate schema and units; reject missing or unexpected inputs explicitly.
- Avoid logging raw sensitive features unless there is a lawful, justified purpose.
- Add request IDs, latency metrics, rate limits, and error monitoring.
- Use a model version field so results can be traced to an artifact.
- Test backwards compatibility before changing the feature contract.

## Practice
Change the request payload to omit one feature. Inspect the validation error. Why is silently filling an unknown missing feature usually unsafe at the API layer?